In [1]:
import os
import json
import math
import random
import contextlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display

from transformers import AutoTokenizer

from config import CONFIG, LOGS_DIR, RAW_RESULTS_DIR
from modes import build_runtime_mode_by_name
from workloads import RuntimeWorkload, BASE_LONG_PROMPT, SHARED_PREFIX
from runner import run_single_benchmark

# Optional HF login from env var
hf_token = os.environ.get("HF_TOKEN", None)
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)

# Lock generation settings for fair benchmarking
CONFIG.generation.do_sample = False
CONFIG.generation.temperature = 0.0
CONFIG.generation.top_p = 1.0
CONFIG.generation.top_k = 0
CONFIG.generation.repetition_penalty = 1.0

# Keep the existing warmup behavior
CONFIG.system.warmup_runs = 1

# Number of measured trials per case for this notebook run
TRIALS = 15   # change to 20 if you want
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("TRIALS =", TRIALS)
print("warmup_runs =", CONFIG.system.warmup_runs)

TRIALS = 15
warmup_runs = 1


In [2]:
tokenizer_name = CONFIG.model.tokenizer_name_or_path or CONFIG.model.model_name_or_path

tok = AutoTokenizer.from_pretrained(
    tokenizer_name,
    trust_remote_code=CONFIG.model.trust_remote_code,
    token=hf_token,
)

if tok.pad_token is None and tok.eos_token is not None:
    tok.pad_token = tok.eos_token

SEP_IDS = tok.encode("\n\n", add_special_tokens=False)

def token_count(text: str) -> int:
    return len(tok.encode(text, add_special_tokens=False))

def build_exactish_prompt(base_text: str, target_tokens: int) -> str:
    """
    Build a prompt that lands very close to the desired token count.
    """
    base_ids = tok.encode(base_text.strip(), add_special_tokens=False)
    if not base_ids:
        raise ValueError("Base text produced no tokens.")

    ids = []
    unit = base_ids + SEP_IDS

    while len(ids) < target_tokens + len(unit):
        ids.extend(unit)

    ids = ids[:target_tokens]
    text = tok.decode(ids, skip_special_tokens=True)

    # Tighten a few times because decode->encode can drift slightly
    for _ in range(8):
        cur_ids = tok.encode(text, add_special_tokens=False)
        if len(cur_ids) == target_tokens:
            return text

        if len(cur_ids) > target_tokens:
            cur_ids = cur_ids[:target_tokens]
        else:
            needed = target_tokens - len(cur_ids)
            filler = unit * ((needed // len(unit)) + 2)
            cur_ids = cur_ids + filler[:needed]

        text = tok.decode(cur_ids, skip_special_tokens=True)

    return text

def build_shared_prefix_pair(target_tokens: int):
    """
    Create a repeated-prefix pair where almost the entire prompt is shared
    and only the tail differs.
    """
    suffix_a = (
        "Task: Summarize when prefix caching helps and what kind of repeated "
        "prompt structure makes it useful."
    )
    suffix_b = (
        "Task: Explain why repeated shared-prefix requests can reduce prefill "
        "cost in a production serving system."
    )

    suffix_a_ids = tok.encode("\n\n" + suffix_a, add_special_tokens=False)
    suffix_b_ids = tok.encode("\n\n" + suffix_b, add_special_tokens=False)

    shared_target = target_tokens - max(len(suffix_a_ids), len(suffix_b_ids)) - 8
    shared_target = max(shared_target, 512)

    shared_ids_unit = tok.encode(SHARED_PREFIX.strip(), add_special_tokens=False) + SEP_IDS
    shared_ids = []

    while len(shared_ids) < shared_target + len(shared_ids_unit):
        shared_ids.extend(shared_ids_unit)

    shared_ids = shared_ids[:shared_target]

    ids_a = shared_ids + suffix_a_ids
    ids_b = shared_ids + suffix_b_ids

    prompt_a = tok.decode(ids_a, skip_special_tokens=True)
    prompt_b = tok.decode(ids_b, skip_special_tokens=True)

    return prompt_a, prompt_b

In [3]:
ctx4032_prompt = build_exactish_prompt(BASE_LONG_PROMPT, 4032)
ctx3840_prompt = build_exactish_prompt(BASE_LONG_PROMPT, 3840)
shared4032_prompt, shared4032_followup = build_shared_prefix_pair(4032)

workloads_4096 = {
    "ctx4032_out64": RuntimeWorkload(
        name="ctx4032_out64",
        prompt=ctx4032_prompt,
        max_new_tokens=64,
        description="4032-token context, 64-token output",
        prompt_tokens_target=4032,
        repeated_prefix=False,
        memory_pressure=True,
        metadata={"custom_test": True, "prompt_tokens_target": 4032, "max_new_tokens": 64},
    ),
    "ctx3840_out256": RuntimeWorkload(
        name="ctx3840_out256",
        prompt=ctx3840_prompt,
        max_new_tokens=256,
        description="3840-token context, 256-token output",
        prompt_tokens_target=3840,
        repeated_prefix=False,
        memory_pressure=True,
        metadata={"custom_test": True, "prompt_tokens_target": 3840, "max_new_tokens": 256},
    ),
    "shared4032_out64": RuntimeWorkload(
        name="shared4032_out64",
        prompt=shared4032_prompt,
        followup_prompt=shared4032_followup,
        max_new_tokens=64,
        description="4032-token shared-prefix test, 64-token output",
        prompt_tokens_target=4032,
        repeated_prefix=True,
        memory_pressure=True,
        metadata={"custom_test": True, "prompt_tokens_target": 4032, "max_new_tokens": 64},
    ),
}

for k, w in workloads_4096.items():
    print(k, "->", token_count(w.prompt), "tokens", "| max_new_tokens =", w.max_new_tokens)
    if w.followup_prompt is not None:
        print("   followup ->", token_count(w.followup_prompt), "tokens")

ctx4032_out64 -> 4032 tokens | max_new_tokens = 64
ctx3840_out256 -> 3840 tokens | max_new_tokens = 256
shared4032_out64 -> 3936 tokens | max_new_tokens = 64
   followup -> 3935 tokens


In [4]:
# Keep fp16_baseline as an optional anchor. Remove it if runtime is too much.
selected_modes = [
    "fp16_baseline",
    "gptq_4bit",
    "int8_quant",
    "speculative_decoding",
    "prefix_caching",
    "continuous_batching",
    "chunked_prefill",
]

mode_to_workloads = {
    "fp16_baseline": ["ctx4032_out64", "ctx3840_out256"],
    "gptq_4bit": ["ctx4032_out64", "ctx3840_out256"],
    "int8_quant": ["ctx4032_out64", "ctx3840_out256"],
    "speculative_decoding": ["ctx4032_out64", "ctx3840_out256"],
    "chunked_prefill": ["ctx4032_out64", "ctx3840_out256"],
    "continuous_batching": ["ctx4032_out64"],   
    "prefix_caching": ["shared4032_out64"],
}

cases = []
for trial_index in range(TRIALS):
    for mode_name in selected_modes:
        for workload_key in mode_to_workloads[mode_name]:
            cases.append({
                "mode_name": mode_name,
                "workload_key": workload_key,
                "trial_index": trial_index,
            })

rng = random.Random(SEED)
rng.shuffle(cases)

print("Total randomized runs:", len(cases))
print("First 10 cases:")
for x in cases[:10]:
    print(x)

Total randomized runs: 180
First 10 cases:
{'mode_name': 'fp16_baseline', 'workload_key': 'ctx3840_out256', 'trial_index': 14}
{'mode_name': 'int8_quant', 'workload_key': 'ctx3840_out256', 'trial_index': 4}
{'mode_name': 'gptq_4bit', 'workload_key': 'ctx3840_out256', 'trial_index': 1}
{'mode_name': 'speculative_decoding', 'workload_key': 'ctx4032_out64', 'trial_index': 3}
{'mode_name': 'fp16_baseline', 'workload_key': 'ctx3840_out256', 'trial_index': 1}
{'mode_name': 'fp16_baseline', 'workload_key': 'ctx4032_out64', 'trial_index': 3}
{'mode_name': 'fp16_baseline', 'workload_key': 'ctx4032_out64', 'trial_index': 5}
{'mode_name': 'gptq_4bit', 'workload_key': 'ctx3840_out256', 'trial_index': 10}
{'mode_name': 'fp16_baseline', 'workload_key': 'ctx3840_out256', 'trial_index': 5}
{'mode_name': 'speculative_decoding', 'workload_key': 'ctx3840_out256', 'trial_index': 1}


In [5]:
run_tag = f"screen_test_4096_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
partial_csv_path = RAW_RESULTS_DIR / f"{run_tag}_partial.csv"
final_csv_path = RAW_RESULTS_DIR / f"{run_tag}_results.csv"
final_json_path = RAW_RESULTS_DIR / f"{run_tag}_results.json"

def safe_name(text: str) -> str:
    return "".join(ch if ch.isalnum() or ch in {"-", "_", "."} else "_" for ch in text)

def fmt_metric(x, digits=2):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "-"
    return f"{x:.{digits}f}"

results_dicts = []

print("=" * 100)
print("Starting", run_tag)
print("Total runs:", len(cases))
print("Logs dir:", LOGS_DIR)
print("=" * 100)

try:
    for idx, case in enumerate(cases, start=1):
        mode_name = case["mode_name"]
        workload_key = case["workload_key"]
        trial_index = case["trial_index"]

        runtime_mode = build_runtime_mode_by_name(mode_name)
        runtime_workload = workloads_4096[workload_key]

        log_path = LOGS_DIR / (
            f"{run_tag}_{idx:04d}_{safe_name(mode_name)}_{safe_name(workload_key)}_trial{trial_index}.log"
        )

        with open(log_path, "w", encoding="utf-8") as log_file:
            with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                result = run_single_benchmark(
                    runtime_mode=runtime_mode,
                    workload=runtime_workload,
                    trial_index=trial_index,
                )

        result.notes += f"log_file={log_path.name}. "
        result_dict = result.to_dict()
        result_dict["workload_key"] = workload_key
        result_dict["log_file"] = log_path.name
        results_dicts.append(result_dict)

        if result.error:
            print(
                f"[{idx}/{len(cases)}] "
                f"{mode_name:<22} | {workload_key:<20} | trial={trial_index:<2} | FAILED | {log_path.name}"
            )
        else:
            print(
                f"[{idx}/{len(cases)}] "
                f"{mode_name:<22} | {workload_key:<20} | trial={trial_index:<2} | "
                f"ttft={fmt_metric(result.ttft_ms)} ms | "
                f"lat={fmt_metric(result.total_latency_ms)} ms | "
                f"tps={fmt_metric(result.tokens_per_second)} | "
                f"J/tok={fmt_metric(result.energy_per_token_j, 3)} | "
                f"gpu={fmt_metric(result.peak_gpu_memory_mb)} MB"
            )

        if idx % 5 == 0 or idx == len(cases):
            pd.DataFrame(results_dicts).to_csv(partial_csv_path, index=False)

except KeyboardInterrupt:
    print("\nInterrupted. Saving partial results...")

finally:
    results_df = pd.DataFrame(results_dicts)
    results_df.to_csv(final_csv_path, index=False)
    with open(final_json_path, "w", encoding="utf-8") as f:
        json.dump(results_dicts, f, indent=2, ensure_ascii=False)

    print("\nSaved:")
    print(final_csv_path)
    print(final_json_path)

Starting screen_test_4096_20260423_102458
Total runs: 180
Logs dir: /scratch/as18181/ModeSwitch-LLM/ModeSwitch-LLM/logs


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


[1/180] fp16_baseline          | ctx3840_out256       | trial=14 | ttft=279.76 ms | lat=5657.73 ms | tps=45.25 | J/tok=4.232 | gpu=33050.93 MB


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


[2/180] int8_quant             | ctx3840_out256       | trial=4  | ttft=214.89 ms | lat=3341.07 ms | tps=76.62 | J/tok=2.583 | gpu=33216.19 MB


...

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


[179/180] int8_quant             | ctx4032_out64        | trial=2  | ttft=224.45 ms | lat=921.73 ms | tps=69.43 | J/tok=2.956 | gpu=33585.08 MB


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


[180/180] speculative_decoding   | ctx3840_out256       | trial=13 | ttft=347.89 ms | lat=2203.55 ms | tps=116.18 | J/tok=1.797 | gpu=33587.21 MB

Saved:
/scratch/as18181/ModeSwitch-LLM/ModeSwitch-LLM/results/raw/screen_test_4096_20260423_102458_results.csv
/scratch/as18181/ModeSwitch-LLM/ModeSwitch-LLM/results/raw/screen_test_4096_20260423_102458_results.json


In [6]:
results_df = pd.read_csv(final_csv_path)

# Keep only successful runs for metric summaries
ok_df = results_df[results_df["success"] == True].copy()

def pctl(x, q):
    x = pd.Series(x).dropna().astype(float)
    if len(x) == 0:
        return np.nan
    return float(np.percentile(x, q))

def mean_ci_bounds(x):
    x = pd.Series(x).dropna().astype(float)
    n = len(x)
    if n == 0:
        return (np.nan, np.nan, np.nan)
    mean = float(x.mean())
    if n == 1:
        return (mean, mean, mean)
    std = float(x.std(ddof=1))
    half = 1.96 * std / math.sqrt(n)
    return (mean, mean - half, mean + half)

def summarize_metric(series, prefix):
    mean, ci_lo, ci_hi = mean_ci_bounds(series)
    s = pd.Series(series).dropna().astype(float)
    return {
        f"{prefix}_mean": mean,
        f"{prefix}_std": float(s.std(ddof=1)) if len(s) > 1 else (0.0 if len(s) == 1 else np.nan),
        f"{prefix}_median": float(s.median()) if len(s) else np.nan,
        f"{prefix}_p95": pctl(s, 95),
        f"{prefix}_ci95_low": ci_lo,
        f"{prefix}_ci95_high": ci_hi,
    }

agg_rows = []
for (mode_name, workload_key), group in ok_df.groupby(["mode_name", "workload_key"], sort=False):
    row = {
        "mode_name": mode_name,
        "workload_key": workload_key,
        "num_runs": len(group),
    }
    row.update(summarize_metric(group["ttft_ms"], "ttft_ms"))
    row.update(summarize_metric(group["total_latency_ms"], "total_latency_ms"))
    row.update(summarize_metric(group["tokens_per_second"], "tokens_per_second"))
    row.update(summarize_metric(group["energy_per_token_j"], "energy_per_token_j"))
    row.update(summarize_metric(group["peak_gpu_memory_mb"], "peak_gpu_memory_mb"))
    agg_rows.append(row)

agg_4096_df = pd.DataFrame(agg_rows)

print("4096 AGGREGATE TABLE")
display(
    agg_4096_df.sort_values(["workload_key", "total_latency_ms_mean"], ascending=[True, True])
)

4096 AGGREGATE TABLE


,mode_name,workload_key,num_runs,ttft_ms_mean,ttft_ms_std,ttft_ms_median,ttft_ms_p95,ttft_ms_ci95_low,ttft_ms_ci95_high,total_latency_ms_mean,...,energy_per_token_j_median,energy_per_token_j_p95,energy_per_token_j_ci95_low,energy_per_token_j_ci95_high,peak_gpu_memory_mb_mean,peak_gpu_memory_mb_std,peak_gpu_memory_mb_median,peak_gpu_memory_mb_p95,peak_gpu_memory_mb_ci95_low,peak_gpu_memory_mb_ci95_high
5,speculative_decoding,ctx3840_out256,15,347.581283,9.549874,350.905374,361.096109,342.748380,352.414186,2196.529065,...,1.797119,1.857773,1.777175,1.816984,33519.218783,67.848200,33552.085449,33562.622949,33484.882856,33553.554709
2,gptq_4bit,ctx3840_out256,15,315.684524,11.551320,315.029508,333.702072,309.838749,321.530299,2462.427763,...,2.003801,2.044962,1.913662,2.000901,33524.150716,86.965737,33563.942383,33563.942383,33480.139981,33568.161451
1,int8_quant,ctx3840_out256,15,219.573244,9.283027,218.093819,232.266142,214.875384,224.271104,3091.210774,...,2.310281,2.456585,2.283813,2.368680,33498.859049,104.732990,33556.192383,33556.192383,33445.856844,33551.861255
8,chunked_prefill,ctx3840_out256,15,308.610545,11.369604,306.077489,322.844848,302.856731,314.364358,4132.645170,...,3.789747,3.830921,3.776418,3.803275,33523.860026,48.426506,33550.943359,33550.943359,33499.352833,33548.367219
0,fp16_baseline,ctx3840_out256,15,276.739055,0.960701,276.432783,277.967724,276.252873,277.225237,5777.719037,...,4.330163,4.405408,4.278851,4.346778,33466.681641,139.891779,33526.556641,33550.931641,33395.886633,33537.476649
6,gptq_4bit,ctx4032_out64,15,322.911895,10.883253,319.004070,344.488941,317.404210,328.419581,787.842936,...,2.857030,3.016391,2.731664,2.895201,33536.696777,59.246477,33570.821777,33578.946777,33506.713923,33566.679632
3,speculative_decoding,ctx4032_out64,15,358.464214,7.791224,360.585725,370.077275,354.521311,362.407118,802.432912,...,2.979842,3.195902,2.894122,3.040849,33515.102083,85.295736,33548.468750,33583.381250,33471.936485,33558.267682
9,int8_quant,ctx4032_out64,15,229.617727,8.159835,230.547786,239.980099,225.488281,233.747174,927.438313,...,2.933425,2.961762,2.880984,2.931477,33551.492350,42.537711,33576.950684,33579.388184,33529.965298,33573.019402
7,chunked_prefill,ctx4032_out64,15,311.862099,10.702592,308.690387,325.286372,306.445840,317.278358,1245.548217,...,4.771770,5.076913,4.716650,4.876125,33549.609993,53.607691,33571.701660,33579.539160,33522.480759,33576.739228
4,fp16_baseline,ctx4032_out64,15,282.056702,0.708764,281.791385,282.908948,281.698017,282.415386,1654.843007,...,5.677295,5.792416,5.481180,5.680281,33528.356608,82.554996,33571.689941,33571.689941,33486.578016,33570.135200


In [7]:
# If fp16_baseline is present, use it as the reference anchor.
# Otherwise fall back to gptq_4bit.
reference_mode = "fp16_baseline" if "fp16_baseline" in ok_df["mode_name"].unique() else "gptq_4bit"
print("Reference mode for paired comparison:", reference_mode)

ref_df = ok_df[ok_df["mode_name"] == reference_mode][
    ["workload_key", "trial_index", "total_latency_ms", "tokens_per_second", "energy_per_token_j"]
].rename(columns={
    "total_latency_ms": "ref_total_latency_ms",
    "tokens_per_second": "ref_tokens_per_second",
    "energy_per_token_j": "ref_energy_per_token_j",
})

pair_df = ok_df.merge(ref_df, on=["workload_key", "trial_index"], how="inner")
pair_df = pair_df[pair_df["mode_name"] != reference_mode].copy()

pair_df["latency_speedup_vs_ref"] = pair_df["ref_total_latency_ms"] / pair_df["total_latency_ms"]
pair_df["throughput_ratio_vs_ref"] = pair_df["tokens_per_second"] / pair_df["ref_tokens_per_second"]
pair_df["energy_ratio_vs_ref"] = pair_df["energy_per_token_j"] / pair_df["ref_energy_per_token_j"]
pair_df["latency_delta_ms_vs_ref"] = pair_df["total_latency_ms"] - pair_df["ref_total_latency_ms"]

pair_rows = []
for (mode_name, workload_key), group in pair_df.groupby(["mode_name", "workload_key"], sort=False):
    row = {
        "mode_name": mode_name,
        "workload_key": workload_key,
        "num_paired_runs": len(group),
    }
    row.update(summarize_metric(group["latency_speedup_vs_ref"], "latency_speedup_vs_ref"))
    row.update(summarize_metric(group["throughput_ratio_vs_ref"], "throughput_ratio_vs_ref"))
    row.update(summarize_metric(group["energy_ratio_vs_ref"], "energy_ratio_vs_ref"))
    row.update(summarize_metric(group["latency_delta_ms_vs_ref"], "latency_delta_ms_vs_ref"))
    pair_rows.append(row)

paired_4096_df = pd.DataFrame(pair_rows)

print("PAIRED COMPARISONS VS", reference_mode)
display(
    paired_4096_df.sort_values(["workload_key", "latency_speedup_vs_ref_mean"], ascending=[True, False])
)

Reference mode for paired comparison: fp16_baseline
PAIRED COMPARISONS VS fp16_baseline


,mode_name,workload_key,num_paired_runs,latency_speedup_vs_ref_mean,latency_speedup_vs_ref_std,latency_speedup_vs_ref_median,latency_speedup_vs_ref_p95,latency_speedup_vs_ref_ci95_low,latency_speedup_vs_ref_ci95_high,throughput_ratio_vs_ref_mean,...,energy_ratio_vs_ref_median,energy_ratio_vs_ref_p95,energy_ratio_vs_ref_ci95_low,energy_ratio_vs_ref_ci95_high,latency_delta_ms_vs_ref_mean,latency_delta_ms_vs_ref_std,latency_delta_ms_vs_ref_median,latency_delta_ms_vs_ref_p95,latency_delta_ms_vs_ref_ci95_low,latency_delta_ms_vs_ref_ci95_high
3,speculative_decoding,ctx3840_out256,15,2.630612,0.040320,2.622035,2.713055,2.610207,2.651016,2.630612,...,0.415696,0.432792,0.410671,0.422935,-3581.189973,71.528617,-3561.188345,-3497.328466,-3617.388447,-3544.991498
1,gptq_4bit,ctx3840_out256,15,2.356920,0.169590,2.284324,2.647048,2.271096,2.442745,2.356920,...,0.462682,0.474510,0.444212,0.463468,-3315.291274,175.426224,-3251.498167,-3136.058972,-3404.069193,-3226.513356
0,int8_quant,ctx3840_out256,15,1.869894,0.044617,1.876571,1.924346,1.847315,1.892474,1.869894,...,0.536722,0.576806,0.529223,0.549724,-2686.508263,95.603846,-2691.344323,-2537.366346,-2734.890485,-2638.126042
6,chunked_prefill,ctx3840_out256,15,1.398204,0.021566,1.402575,1.425212,1.387291,1.409118,1.398204,...,0.880070,0.901025,0.871443,0.886430,-1645.073868,79.550475,-1654.240859,-1522.489836,-1685.331963,-1604.815772
4,gptq_4bit,ctx4032_out64,15,2.100629,0.028641,2.102579,2.137292,2.086134,2.115123,2.100629,...,0.506457,0.553056,0.486554,0.523153,-867.000071,20.467272,-866.974816,-840.843755,-877.357940,-856.642203
2,speculative_decoding,ctx4032_out64,15,2.062481,0.035140,2.049473,2.115243,2.044698,2.080264,2.062481,...,0.527503,0.593767,0.513926,0.551307,-852.410096,24.370104,-846.190903,-824.653544,-864.743070,-840.077121
7,int8_quant,ctx4032_out64,15,1.784495,0.034107,1.783964,1.831974,1.767235,1.801756,1.784495,...,0.519475,0.548327,0.512793,0.529689,-727.404695,27.806133,-724.292461,-692.553186,-741.476539,-713.332850
5,chunked_prefill,ctx4032_out64,15,1.328706,0.022216,1.332802,1.362455,1.317463,1.339948,1.328706,...,0.841019,0.923981,0.839563,0.881306,-409.294790,25.946439,-410.405498,-372.205429,-422.425500,-396.164081
8,continuous_batching,ctx4032_out64,15,0.896920,0.014143,0.893067,0.918204,0.889763,0.904077,2.410471,...,0.606456,0.644134,0.594233,0.619317,190.230009,26.392846,197.578740,221.455953,176.873386,203.586631


In [8]:
decision_cols_agg = [
    "mode_name",
    "workload_key",
    "num_runs",
    "ttft_ms_mean",
    "ttft_ms_ci95_low",
    "ttft_ms_ci95_high",
    "total_latency_ms_mean",
    "total_latency_ms_ci95_low",
    "total_latency_ms_ci95_high",
    "tokens_per_second_mean",
    "tokens_per_second_ci95_low",
    "tokens_per_second_ci95_high",
    "energy_per_token_j_mean",
    "energy_per_token_j_ci95_low",
    "energy_per_token_j_ci95_high",
    "peak_gpu_memory_mb_mean",
]

decision_cols_pair = [
    "mode_name",
    "workload_key",
    "num_paired_runs",
    "latency_speedup_vs_ref_mean",
    "latency_speedup_vs_ref_ci95_low",
    "latency_speedup_vs_ref_ci95_high",
    "throughput_ratio_vs_ref_mean",
    "throughput_ratio_vs_ref_ci95_low",
    "throughput_ratio_vs_ref_ci95_high",
    "energy_ratio_vs_ref_mean",
    "energy_ratio_vs_ref_ci95_low",
    "energy_ratio_vs_ref_ci95_high",
]

print("DECISION TABLE — ABSOLUTE METRICS")
display(
    agg_4096_df[decision_cols_agg].sort_values(["workload_key", "total_latency_ms_mean"], ascending=[True, True])
)

print("DECISION TABLE — PAIRED VS REFERENCE")
display(
    paired_4096_df[decision_cols_pair].sort_values(["workload_key", "latency_speedup_vs_ref_mean"], ascending=[True, False])
)

DECISION TABLE — ABSOLUTE METRICS


,mode_name,workload_key,num_runs,ttft_ms_mean,ttft_ms_ci95_low,ttft_ms_ci95_high,total_latency_ms_mean,total_latency_ms_ci95_low,total_latency_ms_ci95_high,tokens_per_second_mean,tokens_per_second_ci95_low,tokens_per_second_ci95_high,energy_per_token_j_mean,energy_per_token_j_ci95_low,energy_per_token_j_ci95_high,peak_gpu_memory_mb_mean
5,speculative_decoding,ctx3840_out256,15,347.581283,342.748380,352.414186,2196.529065,2186.045483,2207.012646,116.557155,116.002777,117.111534,1.797079,1.777175,1.816984,33519.218783
2,gptq_4bit,ctx3840_out256,15,315.684524,309.838749,321.530299,2462.427763,2378.050366,2546.805161,104.437150,100.633232,108.241067,1.957282,1.913662,2.000901,33524.150716
1,int8_quant,ctx3840_out256,15,219.573244,214.875384,224.271104,3091.210774,3055.907899,3126.513650,82.852139,81.970456,83.733822,2.326246,2.283813,2.368680,33498.859049
8,chunked_prefill,ctx3840_out256,15,308.610545,302.856731,314.364358,4132.645170,4110.734407,4154.555933,61.952008,61.630513,62.273503,3.789847,3.776418,3.803275,33523.860026
0,fp16_baseline,ctx3840_out256,15,276.739055,276.252873,277.225237,5777.719037,5743.375112,5812.062963,44.313801,44.052575,44.575027,4.312814,4.278851,4.346778,33466.681641
6,gptq_4bit,ctx4032_out64,15,322.911895,317.404210,328.419581,787.842936,782.439386,793.246486,81.248281,80.695518,81.801045,2.813433,2.731664,2.895201,33536.696777
3,speculative_decoding,ctx4032_out64,15,358.464214,354.521311,362.407118,802.432912,797.850689,807.015134,79.766928,79.311292,80.222564,2.967485,2.894122,3.040849,33515.102083
9,int8_quant,ctx4032_out64,15,229.617727,225.488281,233.747174,927.438313,923.803472,931.073154,69.011172,68.739053,69.283290,2.906230,2.880984,2.931477,33551.492350
7,chunked_prefill,ctx4032_out64,15,311.862099,306.445840,317.278358,1245.548217,1239.550550,1251.545885,51.387333,51.140197,51.634470,4.796388,4.716650,4.876125,33549.609993
4,fp16_baseline,ctx4032_out64,15,282.056702,281.698017,282.415386,1654.843007,1642.696590,1666.989425,38.681926,38.399173,38.964679,5.580730,5.481180,5.680281,33528.356608


DECISION TABLE — PAIRED VS REFERENCE


,mode_name,workload_key,num_paired_runs,latency_speedup_vs_ref_mean,latency_speedup_vs_ref_ci95_low,latency_speedup_vs_ref_ci95_high,throughput_ratio_vs_ref_mean,throughput_ratio_vs_ref_ci95_low,throughput_ratio_vs_ref_ci95_high,energy_ratio_vs_ref_mean,energy_ratio_vs_ref_ci95_low,energy_ratio_vs_ref_ci95_high
3,speculative_decoding,ctx3840_out256,15,2.630612,2.610207,2.651016,2.630612,2.610207,2.651016,0.416803,0.410671,0.422935
1,gptq_4bit,ctx3840_out256,15,2.356920,2.271096,2.442745,2.356920,2.271096,2.442745,0.453840,0.444212,0.463468
0,int8_quant,ctx3840_out256,15,1.869894,1.847315,1.892474,1.869894,1.847315,1.892474,0.539473,0.529223,0.549724
6,chunked_prefill,ctx3840_out256,15,1.398204,1.387291,1.409118,1.398204,1.387291,1.409118,0.878936,0.871443,0.886430
4,gptq_4bit,ctx4032_out64,15,2.100629,2.086134,2.115123,2.100629,2.086134,2.115123,0.504854,0.486554,0.523153
2,speculative_decoding,ctx4032_out64,15,2.062481,2.044698,2.080264,2.062481,2.044698,2.080264,0.532616,0.513926,0.551307
7,int8_quant,ctx4032_out64,15,1.784495,1.767235,1.801756,1.784495,1.767235,1.801756,0.521241,0.512793,0.529689
5,chunked_prefill,ctx4032_out64,15,1.328706,1.317463,1.339948,1.328706,1.317463,1.339948,0.860434,0.839563,0.881306
8,continuous_batching,ctx4032_out64,15,0.896920,0.889763,0.904077,2.410471,2.391237,2.429706,0.606775,0.594233,0.619317


In [9]:
spot_df = ok_df.sort_values(["mode_name", "workload_key", "trial_index"]).groupby(
    ["mode_name", "workload_key"], as_index=False
).head(1)[["mode_name", "workload_key", "trial_index", "output_text"]]

for _, row in spot_df.iterrows():
    print("=" * 100)
    print(row["mode_name"], "|", row["workload_key"], "| trial", int(row["trial_index"]))
    print("-" * 100)
    print(str(row["output_text"])[:1200])
    print()

chunked_prefill | ctx3840_out256 | trial 0
----------------------------------------------------------------------------------------------------
follows.

Large language model inference often consists of two major phases: prefill and decode. The prefill stage processes the input prompt and builds the key-value cache, while the decode stage generates output tokens one at a time. These two phases can stress hardware differently. Prefill is often more compute-intensive because it processes many prompt tokens together, while decode can become more memory-bandwidth-bound because the model repeatedly reads and updates cached activations over many steps. Practical efficiency techniques may therefore help one phase more than the other. Quantization can reduce memory footprint, speculative decoding can reduce per-token latency, prefix caching can avoid recomputing shared prompt segments, and chunked prefill can improve scheduling behavior when prompts are long. Because of this, a fixed inference

# Hybrid Modes

In [19]:
from modes import build_hybrid_mode

In [20]:
# IMPORTANT:
# With max_model_len=4096, keep prompt+generation within budget.

ctx4032_prompt = build_exactish_prompt(BASE_LONG_PROMPT, 4032)
ctx3840_prompt = build_exactish_prompt(BASE_LONG_PROMPT, 3840)
ctx2048_prompt = build_exactish_prompt(BASE_LONG_PROMPT, 2048)
shared4032_prompt, shared4032_followup = build_shared_prefix_pair(4032)

hybrid_workloads = {
    "ctx4032_out64": RuntimeWorkload(
        name="ctx4032_out64",
        prompt=ctx4032_prompt,
        max_new_tokens=64,
        description="4032-token context, 64-token output",
        prompt_tokens_target=4032,
        repeated_prefix=False,
        memory_pressure=True,
        metadata={"custom_test": True, "prompt_tokens_target": 4032, "max_new_tokens": 64},
    ),
    "ctx3840_out256": RuntimeWorkload(
        name="ctx3840_out256",
        prompt=ctx3840_prompt,
        max_new_tokens=256,
        description="3840-token context, 256-token output",
        prompt_tokens_target=3840,
        repeated_prefix=False,
        memory_pressure=True,
        metadata={"custom_test": True, "prompt_tokens_target": 3840, "max_new_tokens": 256},
    ),
    "ctx2048_out64": RuntimeWorkload(
        name="ctx2048_out64",
        prompt=ctx2048_prompt,
        max_new_tokens=64,
        description="2048-token context, 64-token output",
        prompt_tokens_target=2048,
        repeated_prefix=False,
        memory_pressure=False,
        metadata={"custom_test": True, "prompt_tokens_target": 2048, "max_new_tokens": 64},
    ),
    "shared4032_out64": RuntimeWorkload(
        name="shared4032_out64",
        prompt=shared4032_prompt,
        followup_prompt=shared4032_followup,
        max_new_tokens=64,
        description="4032-token shared-prefix workload, 64-token output",
        prompt_tokens_target=4032,
        repeated_prefix=True,
        memory_pressure=True,
        metadata={"custom_test": True, "prompt_tokens_target": 4032, "max_new_tokens": 64},
    ),
}

for k, w in hybrid_workloads.items():
    print(k, "->", token_count(w.prompt), "tokens", "| max_new_tokens =", w.max_new_tokens)
    if w.followup_prompt is not None:
        print("   followup ->", token_count(w.followup_prompt), "tokens")
    prompt_budget = token_count(w.prompt)
    if w.followup_prompt is not None:
        prompt_budget = max(prompt_budget, token_count(w.followup_prompt))
    total_budget = prompt_budget + w.max_new_tokens
    print("   total budget ->", total_budget)
    assert total_budget <= CONFIG.model.max_model_len, (
        f"{k} exceeds max_model_len: {total_budget} > {CONFIG.model.max_model_len}"
    )

ctx4032_out64 -> 4032 tokens | max_new_tokens = 64
   total budget -> 4096
ctx3840_out256 -> 3840 tokens | max_new_tokens = 256
   total budget -> 4096
ctx2048_out64 -> 2048 tokens | max_new_tokens = 64
   total budget -> 2112
shared4032_out64 -> 3936 tokens | max_new_tokens = 64
   followup -> 3935 tokens
   total budget -> 4000


In [21]:
# Smart hybrids only.
# I am NOT adding cuda_graphs hybrids because most non-baseline vLLM runs are
# already non-eager and would make that hybrid less informative.

hybrid_modes = {
    "gptq_plus_chunked_prefill": build_hybrid_mode(
        name="gptq_plus_chunked_prefill",
        base_mode_name="gptq_4bit",
        extra_flags={
            "chunked_prefill": True,
            "max_num_batched_tokens": 4096,
        },
        description="GPTQ + chunked prefill",
        primary_phase="both",
    ),
    "int8_plus_chunked_prefill": build_hybrid_mode(
        name="int8_plus_chunked_prefill",
        base_mode_name="int8_quant",
        extra_flags={
            "chunked_prefill": True,
            "max_num_batched_tokens": 4096,
        },
        description="INT8 + chunked prefill",
        primary_phase="both",
    ),
    "speculative_plus_chunked_prefill": build_hybrid_mode(
        name="speculative_plus_chunked_prefill",
        base_mode_name="speculative_decoding",
        extra_flags={
            "chunked_prefill": True,
            "max_num_batched_tokens": 4096,
        },
        description="Speculative decoding + chunked prefill",
        primary_phase="both",
    ),
    "gptq_plus_prefix_caching": build_hybrid_mode(
        name="gptq_plus_prefix_caching",
        base_mode_name="gptq_4bit",
        extra_flags={
            "prefix_caching": True,
        },
        description="GPTQ + prefix caching",
        primary_phase="both",
    ),
    "int8_plus_prefix_caching": build_hybrid_mode(
        name="int8_plus_prefix_caching",
        base_mode_name="int8_quant",
        extra_flags={
            "prefix_caching": True,
        },
        description="INT8 + prefix caching",
        primary_phase="both",
    ),
    "gptq_plus_continuous_batching": build_hybrid_mode(
        name="gptq_plus_continuous_batching",
        base_mode_name="gptq_4bit",
        extra_flags={
            "continuous_batching": True,
            "max_num_seqs": 4,
            "max_num_batched_tokens": 4096,
        },
        description="GPTQ + continuous batching",
        primary_phase="decode",
    ),
    "int8_plus_continuous_batching": build_hybrid_mode(
        name="int8_plus_continuous_batching",
        base_mode_name="int8_quant",
        extra_flags={
            "continuous_batching": True,
            "max_num_seqs": 4,
            "max_num_batched_tokens": 4096,
        },
        description="INT8 + continuous batching",
        primary_phase="decode",
    ),
}

print("Hybrid modes:")
for k, v in hybrid_modes.items():
    print(k, "->", v.notes)

Hybrid modes:
gptq_plus_chunked_prefill -> quantization=gptq; chunked prefill enabled
int8_plus_chunked_prefill -> quantization=compressed-tensors; chunked prefill enabled
speculative_plus_chunked_prefill -> speculative decoding enabled; chunked prefill enabled
gptq_plus_prefix_caching -> quantization=gptq; prefix caching enabled
int8_plus_prefix_caching -> quantization=compressed-tensors; prefix caching enabled
gptq_plus_continuous_batching -> quantization=gptq; continuous batching benchmark uses a multi-request batch
int8_plus_continuous_batching -> quantization=compressed-tensors; continuous batching benchmark uses a multi-request batch


In [22]:
HYBRID_TRIALS = 10   # 10 is a good overnight number

hybrid_mode_to_workloads = {
    "gptq_plus_chunked_prefill": ["ctx4032_out64", "ctx3840_out256"],
    "int8_plus_chunked_prefill": ["ctx4032_out64", "ctx3840_out256"],
    "speculative_plus_chunked_prefill": ["ctx4032_out64", "ctx3840_out256"],
    "gptq_plus_prefix_caching": ["shared4032_out64"],
    "int8_plus_prefix_caching": ["shared4032_out64"],
    "gptq_plus_continuous_batching": ["ctx2048_out64"],   # keep batching hybrid lighter
    "int8_plus_continuous_batching": ["ctx2048_out64"],   # keep batching hybrid lighter
}

hybrid_cases = []
for trial_index in range(HYBRID_TRIALS):
    for mode_name, workload_keys in hybrid_mode_to_workloads.items():
        for workload_key in workload_keys:
            hybrid_cases.append({
                "mode_name": mode_name,
                "workload_key": workload_key,
                "trial_index": trial_index,
            })

rng = random.Random(SEED + 100)
rng.shuffle(hybrid_cases)

print("Total hybrid runs:", len(hybrid_cases))
print("First 10 hybrid cases:")
for x in hybrid_cases[:10]:
    print(x)

Total hybrid runs: 100
First 10 hybrid cases:
{'mode_name': 'int8_plus_chunked_prefill', 'workload_key': 'ctx3840_out256', 'trial_index': 6}
{'mode_name': 'int8_plus_chunked_prefill', 'workload_key': 'ctx4032_out64', 'trial_index': 9}
{'mode_name': 'gptq_plus_prefix_caching', 'workload_key': 'shared4032_out64', 'trial_index': 1}
{'mode_name': 'int8_plus_chunked_prefill', 'workload_key': 'ctx3840_out256', 'trial_index': 2}
{'mode_name': 'int8_plus_prefix_caching', 'workload_key': 'shared4032_out64', 'trial_index': 2}
{'mode_name': 'gptq_plus_chunked_prefill', 'workload_key': 'ctx3840_out256', 'trial_index': 4}
{'mode_name': 'speculative_plus_chunked_prefill', 'workload_key': 'ctx4032_out64', 'trial_index': 1}
{'mode_name': 'gptq_plus_prefix_caching', 'workload_key': 'shared4032_out64', 'trial_index': 4}
{'mode_name': 'gptq_plus_chunked_prefill', 'workload_key': 'ctx4032_out64', 'trial_index': 8}
{'mode_name': 'int8_plus_continuous_batching', 'workload_key': 'ctx2048_out64', 'trial_index

In [23]:
hybrid_run_tag = f"hybrid_screen_test_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
hybrid_partial_csv_path = RAW_RESULTS_DIR / f"{hybrid_run_tag}_partial.csv"
hybrid_final_csv_path = RAW_RESULTS_DIR / f"{hybrid_run_tag}_results.csv"
hybrid_final_json_path = RAW_RESULTS_DIR / f"{hybrid_run_tag}_results.json"

hybrid_results_dicts = []

print("=" * 100)
print("Starting", hybrid_run_tag)
print("Total hybrid runs:", len(hybrid_cases))
print("Logs dir:", LOGS_DIR)
print("=" * 100)

try:
    for idx, case in enumerate(hybrid_cases, start=1):
        mode_name = case["mode_name"]
        workload_key = case["workload_key"]
        trial_index = case["trial_index"]

        runtime_mode = hybrid_modes[mode_name]
        runtime_workload = hybrid_workloads[workload_key]

        log_path = LOGS_DIR / (
            f"{hybrid_run_tag}_{idx:04d}_{safe_name(mode_name)}_{safe_name(workload_key)}_trial{trial_index}.log"
        )

        with open(log_path, "w", encoding="utf-8") as log_file:
            with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                result = run_single_benchmark(
                    runtime_mode=runtime_mode,
                    workload=runtime_workload,
                    trial_index=trial_index,
                )

        result.notes += f"log_file={log_path.name}. "
        result_dict = result.to_dict()
        result_dict["workload_key"] = workload_key
        result_dict["log_file"] = log_path.name
        hybrid_results_dicts.append(result_dict)

        if result.error:
            print(
                f"[{idx}/{len(hybrid_cases)}] "
                f"{mode_name:<30} | {workload_key:<18} | trial={trial_index:<2} | FAILED | {log_path.name}"
            )
        else:
            print(
                f"[{idx}/{len(hybrid_cases)}] "
                f"{mode_name:<30} | {workload_key:<18} | trial={trial_index:<2} | "
                f"ttft={fmt_metric(result.ttft_ms)} ms | "
                f"lat={fmt_metric(result.total_latency_ms)} ms | "
                f"tps={fmt_metric(result.tokens_per_second)} | "
                f"J/tok={fmt_metric(result.energy_per_token_j, 3)} | "
                f"gpu={fmt_metric(result.peak_gpu_memory_mb)} MB"
            )

        if idx % 5 == 0 or idx == len(hybrid_cases):
            pd.DataFrame(hybrid_results_dicts).to_csv(hybrid_partial_csv_path, index=False)

except KeyboardInterrupt:
    print("\nInterrupted. Saving partial hybrid results...")

finally:
    hybrid_results_df = pd.DataFrame(hybrid_results_dicts)
    hybrid_results_df.to_csv(hybrid_final_csv_path, index=False)
    with open(hybrid_final_json_path, "w", encoding="utf-8") as f:
        json.dump(hybrid_results_dicts, f, indent=2, ensure_ascii=False)

    print("\nSaved hybrid results:")
    print(hybrid_final_csv_path)
    print(hybrid_final_json_path)

Starting hybrid_screen_test_20260423_111519
Total hybrid runs: 100
Logs dir: /scratch/as18181/ModeSwitch-LLM/ModeSwitch-LLM/logs


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


[1/100] int8_plus_chunked_prefill      | ctx3840_out256     | trial=6  | ttft=238.23 ms | lat=3122.84 ms | tps=81.98 | J/tok=2.312 | gpu=33564.32 MB


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


[2/100] int8_plus_chunked_prefill      | ctx4032_out64      | trial=9  | ttft=220.80 ms | lat=922.41 ms | tps=69.38 | J/tok=3.204 | gpu=33585.08 MB


...

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


[99/100] gptq_plus_continuous_batching  | ctx2048_out64      | trial=8  | ttft=343.91 ms | lat=1276.24 ms | tps=200.59 | J/tok=1.468 | gpu=33487.54 MB


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


[100/100] speculative_plus_chunked_prefill | ctx4032_out64      | trial=7  | ttft=368.55 ms | lat=808.44 ms | tps=79.16 | J/tok=2.882 | gpu=33607.97 MB

Saved hybrid results:
/scratch/as18181/ModeSwitch-LLM/ModeSwitch-LLM/results/raw/hybrid_screen_test_20260423_111519_results.csv
/scratch/as18181/ModeSwitch-LLM/ModeSwitch-LLM/results/raw/hybrid_screen_test_20260423_111519_results.json


In [24]:
hybrid_results_df = pd.read_csv(hybrid_final_csv_path)
hybrid_ok_df = hybrid_results_df[hybrid_results_df["success"] == True].copy()

hybrid_agg_rows = []
for (mode_name, workload_key), group in hybrid_ok_df.groupby(["mode_name", "workload_key"], sort=False):
    row = {
        "mode_name": mode_name,
        "workload_key": workload_key,
        "num_runs": len(group),
    }
    row.update(summarize_metric(group["ttft_ms"], "ttft_ms"))
    row.update(summarize_metric(group["total_latency_ms"], "total_latency_ms"))
    row.update(summarize_metric(group["tokens_per_second"], "tokens_per_second"))
    row.update(summarize_metric(group["energy_per_token_j"], "energy_per_token_j"))
    row.update(summarize_metric(group["peak_gpu_memory_mb"], "peak_gpu_memory_mb"))
    hybrid_agg_rows.append(row)

hybrid_agg_df = pd.DataFrame(hybrid_agg_rows)

print("HYBRID AGGREGATE TABLE")
display(
    hybrid_agg_df.sort_values(["workload_key", "total_latency_ms_mean"], ascending=[True, True])
)

HYBRID AGGREGATE TABLE


,mode_name,workload_key,num_runs,ttft_ms_mean,ttft_ms_std,ttft_ms_median,ttft_ms_p95,ttft_ms_ci95_low,ttft_ms_ci95_high,total_latency_ms_mean,...,energy_per_token_j_median,energy_per_token_j_p95,energy_per_token_j_ci95_low,energy_per_token_j_ci95_high,peak_gpu_memory_mb_mean,peak_gpu_memory_mb_std,peak_gpu_memory_mb_median,peak_gpu_memory_mb_p95,peak_gpu_memory_mb_ci95_low,peak_gpu_memory_mb_ci95_high
7,int8_plus_continuous_batching,ctx2048_out64,10,234.742493,6.914066,232.855107,246.447729,230.457110,239.027875,1203.919103,...,1.085437,1.138801,1.055699,1.109484,33377.311523,0.0,33377.311523,33377.311523,33377.311523,33377.311523
8,gptq_plus_continuous_batching,ctx2048_out64,10,344.721096,8.145381,345.205010,354.844621,339.672537,349.769656,1279.769378,...,1.437064,1.505711,1.403541,1.466284,33487.538086,0.0,33487.538086,33487.538086,33487.538086,33487.538086
9,speculative_plus_chunked_prefill,ctx3840_out256,10,352.571661,7.510350,354.737288,360.600395,347.916698,357.226624,2203.971474,...,1.809497,1.869104,1.763249,1.835059,33587.211426,0.0,33587.211426,33587.211426,33587.211426,33587.211426
4,gptq_plus_chunked_prefill,ctx3840_out256,10,317.226280,12.704019,318.348907,335.079608,309.352247,325.100312,2214.970552,...,1.821096,1.916679,1.786302,1.864845,33625.943359,0.0,33625.943359,33625.943359,33625.943359,33625.943359
0,int8_plus_chunked_prefill,ctx3840_out256,10,226.571440,8.348656,226.498686,237.394798,221.396889,231.745991,3095.273840,...,2.296398,2.330255,2.273498,2.310898,33564.318359,0.0,33564.318359,33564.318359,33564.318359,33564.318359
6,gptq_plus_chunked_prefill,ctx4032_out64,10,322.989967,9.472861,324.648918,334.025666,317.118626,328.861307,788.215273,...,2.828007,2.968543,2.755982,2.911669,33640.947754,0.0,33640.947754,33640.947754,33640.947754,33640.947754
5,speculative_plus_chunked_prefill,ctx4032_out64,10,361.943531,12.302790,363.774000,377.904238,354.318183,369.568879,807.130791,...,2.917812,3.133383,2.867615,3.021370,33607.969727,0.0,33607.969727,33607.969727,33607.969727,33607.969727
1,int8_plus_chunked_prefill,ctx4032_out64,10,233.316081,7.876112,235.267009,243.202753,228.434417,238.197746,934.142434,...,3.014707,3.198479,2.949603,3.093098,33585.076660,0.0,33585.076660,33585.076660,33585.076660,33585.076660
2,gptq_plus_prefix_caching,shared4032_out64,10,54.320555,0.674917,54.308728,55.329280,53.902237,54.738873,541.359981,...,2.098133,2.216443,1.955011,2.143714,33634.139160,0.0,33634.139160,33634.139160,33634.139160,33634.139160
3,int8_plus_prefix_caching,shared4032_out64,10,49.615741,0.490206,49.590117,50.323932,49.311908,49.919574,772.463220,...,2.443211,2.540564,2.359809,2.491272,33574.514160,0.0,33574.514160,33574.514160,33574.514160,33574.514160


In [25]:
# If you already ran the earlier 4096 notebook cells, results_df should exist.
# This combines the old and new runs into one comparison view.

frames = []
if "results_df" in globals():
    frames.append(results_df)
frames.append(hybrid_results_df)

combined_df = pd.concat(frames, ignore_index=True, sort=False)
combined_ok_df = combined_df[combined_df["success"] == True].copy()

combined_agg_rows = []
for (mode_name, workload_key), group in combined_ok_df.groupby(["mode_name", "workload_key"], sort=False):
    row = {
        "mode_name": mode_name,
        "workload_key": workload_key,
        "num_runs": len(group),
    }
    row.update(summarize_metric(group["ttft_ms"], "ttft_ms"))
    row.update(summarize_metric(group["total_latency_ms"], "total_latency_ms"))
    row.update(summarize_metric(group["tokens_per_second"], "tokens_per_second"))
    row.update(summarize_metric(group["energy_per_token_j"], "energy_per_token_j"))
    row.update(summarize_metric(group["peak_gpu_memory_mb"], "peak_gpu_memory_mb"))
    combined_agg_rows.append(row)

combined_agg_df = pd.DataFrame(combined_agg_rows)

print("COMBINED BASE + HYBRID TABLE")
display(
    combined_agg_df.sort_values(["workload_key", "total_latency_ms_mean"], ascending=[True, True])
)

COMBINED BASE + HYBRID TABLE


,mode_name,workload_key,num_runs,ttft_ms_mean,ttft_ms_std,ttft_ms_median,ttft_ms_p95,ttft_ms_ci95_low,ttft_ms_ci95_high,total_latency_ms_mean,...,energy_per_token_j_median,energy_per_token_j_p95,energy_per_token_j_ci95_low,energy_per_token_j_ci95_high,peak_gpu_memory_mb_mean,peak_gpu_memory_mb_std,peak_gpu_memory_mb_median,peak_gpu_memory_mb_p95,peak_gpu_memory_mb_ci95_low,peak_gpu_memory_mb_ci95_high
19,int8_plus_continuous_batching,ctx2048_out64,10,234.742493,6.914066,232.855107,246.447729,230.457110,239.027875,1203.919103,...,1.085437,1.138801,1.055699,1.109484,33377.311523,0.000000,33377.311523,33377.311523,33377.311523,33377.311523
20,gptq_plus_continuous_batching,ctx2048_out64,10,344.721096,8.145381,345.205010,354.844621,339.672537,349.769656,1279.769378,...,1.437064,1.505711,1.403541,1.466284,33487.538086,0.000000,33487.538086,33487.538086,33487.538086,33487.538086
5,speculative_decoding,ctx3840_out256,15,347.581283,9.549874,350.905374,361.096109,342.748380,352.414186,2196.529065,...,1.797119,1.857773,1.777175,1.816984,33519.218783,67.848200,33552.085449,33562.622949,33484.882856,33553.554709
21,speculative_plus_chunked_prefill,ctx3840_out256,10,352.571661,7.510350,354.737288,360.600395,347.916698,357.226624,2203.971474,...,1.809497,1.869104,1.763249,1.835059,33587.211426,0.000000,33587.211426,33587.211426,33587.211426,33587.211426
16,gptq_plus_chunked_prefill,ctx3840_out256,10,317.226280,12.704019,318.348907,335.079608,309.352247,325.100312,2214.970552,...,1.821096,1.916679,1.786302,1.864845,33625.943359,0.000000,33625.943359,33625.943359,33625.943359,33625.943359
2,gptq_4bit,ctx3840_out256,15,315.684524,11.551320,315.029508,333.702072,309.838749,321.530299,2462.427763,...,2.003801,2.044962,1.913662,2.000901,33524.150716,86.965737,33563.942383,33563.942383,33480.139981,33568.161451
1,int8_quant,ctx3840_out256,15,219.573244,9.283027,218.093819,232.266142,214.875384,224.271104,3091.210774,...,2.310281,2.456585,2.283813,2.368680,33498.859049,104.732990,33556.192383,33556.192383,33445.856844,33551.861255
12,int8_plus_chunked_prefill,ctx3840_out256,10,226.571440,8.348656,226.498686,237.394798,221.396889,231.745991,3095.273840,...,2.296398,2.330255,2.273498,2.310898,33564.318359,0.000000,33564.318359,33564.318359,33564.318359,33564.318359
8,chunked_prefill,ctx3840_out256,15,308.610545,11.369604,306.077489,322.844848,302.856731,314.364358,4132.645170,...,3.789747,3.830921,3.776418,3.803275,33523.860026,48.426506,33550.943359,33550.943359,33499.352833,33548.367219
0,fp16_baseline,ctx3840_out256,15,276.739055,0.960701,276.432783,277.967724,276.252873,277.225237,5777.719037,...,4.330163,4.405408,4.278851,4.346778,33466.681641,139.891779,33526.556641,33550.931641,33395.886633,33537.476649


In [26]:
hybrid_decision_cols = [
    "mode_name",
    "workload_key",
    "num_runs",
    "ttft_ms_mean",
    "ttft_ms_ci95_low",
    "ttft_ms_ci95_high",
    "total_latency_ms_mean",
    "total_latency_ms_ci95_low",
    "total_latency_ms_ci95_high",
    "tokens_per_second_mean",
    "tokens_per_second_ci95_low",
    "tokens_per_second_ci95_high",
    "energy_per_token_j_mean",
    "energy_per_token_j_ci95_low",
    "energy_per_token_j_ci95_high",
    "peak_gpu_memory_mb_mean",
]

print("HYBRID DECISION TABLE")
display(
    hybrid_agg_df[hybrid_decision_cols].sort_values(
        ["workload_key", "total_latency_ms_mean"],
        ascending=[True, True],
    )
)

HYBRID DECISION TABLE


,mode_name,workload_key,num_runs,ttft_ms_mean,ttft_ms_ci95_low,ttft_ms_ci95_high,total_latency_ms_mean,total_latency_ms_ci95_low,total_latency_ms_ci95_high,tokens_per_second_mean,tokens_per_second_ci95_low,tokens_per_second_ci95_high,energy_per_token_j_mean,energy_per_token_j_ci95_low,energy_per_token_j_ci95_high,peak_gpu_memory_mb_mean
7,int8_plus_continuous_batching,ctx2048_out64,10,234.742493,230.457110,239.027875,1203.919103,1200.091842,1207.746364,212.643896,211.969303,213.318489,1.082592,1.055699,1.109484,33377.311523
8,gptq_plus_continuous_batching,ctx2048_out64,10,344.721096,339.672537,349.769656,1279.769378,1274.596923,1284.941832,200.043673,199.237663,200.849683,1.434913,1.403541,1.466284,33487.538086
9,speculative_plus_chunked_prefill,ctx3840_out256,10,352.571661,347.916698,357.226624,2203.971474,2198.908356,2209.034593,116.155390,115.888481,116.422299,1.799154,1.763249,1.835059,33587.211426
4,gptq_plus_chunked_prefill,ctx3840_out256,10,317.226280,309.352247,325.100312,2214.970552,2206.283113,2223.657991,115.581323,115.128002,116.034643,1.825574,1.786302,1.864845,33625.943359
0,int8_plus_chunked_prefill,ctx3840_out256,10,226.571440,221.396889,231.745991,3095.273840,3085.377020,3105.170660,82.708715,82.444616,82.972815,2.292198,2.273498,2.310898,33564.318359
6,gptq_plus_chunked_prefill,ctx4032_out64,10,322.989967,317.118626,328.861307,788.215273,781.981933,794.448613,81.208067,80.561680,81.854455,2.833825,2.755982,2.911669,33640.947754
5,speculative_plus_chunked_prefill,ctx4032_out64,10,361.943531,354.318183,369.568879,807.130791,799.644817,814.616765,79.309283,78.569990,80.048576,2.944492,2.867615,3.021370,33607.969727
1,int8_plus_chunked_prefill,ctx4032_out64,10,233.316081,228.434417,238.197746,934.142434,929.096210,939.188658,68.516732,68.145690,68.887773,3.021350,2.949603,3.093098,33585.076660
2,gptq_plus_prefix_caching,shared4032_out64,10,54.320555,53.902237,54.738873,541.359981,538.137651,544.582311,118.230482,117.534835,118.926129,2.049363,1.955011,2.143714,33634.139160
3,int8_plus_prefix_caching,shared4032_out64,10,49.615741,49.311908,49.919574,772.463220,770.545858,774.380581,82.853037,82.647343,83.058732,2.425540,2.359809,2.491272,33574.514160
